In [1]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=10.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 传入 'uniform'
        self.total_epochs = epochs
        
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))
        
        # 🔥 新增：设定热身轮数
        self.warmup_epochs = 3 

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 🔥 修改 1：前三轮热身期，使用极低的噪声系数 (例如 0.05)
        if current_epoch < self.warmup_epochs:
            return 0.15 

        # 正常退火期：从 1.0 降到 0.1
        min_decay = 0.1 
        
        # 🔥 修改 2：扣除 warmup 轮数来计算实际进度，保证后续退火曲线完整平滑
        progress = (current_epoch - self.warmup_epochs) / max(1, (self.total_epochs - self.warmup_epochs))
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        # 🔥 修改 3：移除 if current_epoch < 3: return，让热身期也能执行 AGC 和低噪声注入

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC (自适应梯度裁剪)
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印 (🔥 优化了 Log，可以直接看出目前是热身还是退火)
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            phase = "🔥 Warmup" if current_epoch < self.warmup_epochs else "📉 Annealing"
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} ({phase}) | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=10.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp final: 修正")
print("✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 + 前3轮低噪声热身")

if os.path.exists(PRETRAINED_GC10):
    trainer_rescue = Trainer_Rescue(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': 'Final',
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
    })
    trainer_rescue.train()
else:
    print("❌ 没找到 Exp 13 的权重，无法执行拯救计划。")

🚀 开始 Exp final: 修正
✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 + 前3轮低噪声热身
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False

In [2]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=10.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 传入 'uniform'
        self.total_epochs = epochs
        
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))
        
        # 🔥 新增：设定热身轮数
        self.warmup_epochs = 3 

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 🔥 修改 1：前三轮热身期，使用极低的噪声系数 (例如 0.05)
        if current_epoch < self.warmup_epochs:
            return 0.10 

        # 正常退火期：从 1.0 降到 0.1
        min_decay = 0.1 
        
        # 🔥 修改 2：扣除 warmup 轮数来计算实际进度，保证后续退火曲线完整平滑
        progress = (current_epoch - self.warmup_epochs) / max(1, (self.total_epochs - self.warmup_epochs))
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        # 🔥 修改 3：移除 if current_epoch < 3: return，让热身期也能执行 AGC 和低噪声注入

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC (自适应梯度裁剪)
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印 (🔥 优化了 Log，可以直接看出目前是热身还是退火)
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            phase = "🔥 Warmup" if current_epoch < self.warmup_epochs else "📉 Annealing"
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} ({phase}) | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=10.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp final: 修正")
print("✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 + 前3轮低噪声热身")

if os.path.exists(PRETRAINED_GC10):
    trainer_rescue = Trainer_Rescue(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 50,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': 'Final_v1',
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
    })
    trainer_rescue.train()
else:
    print("❌ 没找到 Exp 13 的权重，无法执行拯救计划。")

🚀 开始 Exp final: 修正
✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 + 前3轮低噪声热身
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False

In [3]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=10.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 传入 'uniform'
        self.total_epochs = epochs
        
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))
        
        # 🔥 新增：设定热身轮数
        self.warmup_epochs = 3 

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 🔥 修改 1：前三轮热身期，使用极低的噪声系数 (例如 0.05)
        if current_epoch < self.warmup_epochs:
            return 0.15 

        # 正常退火期：从 1.0 降到 0.1
        min_decay = 0.1 
        
        # 🔥 修改 2：扣除 warmup 轮数来计算实际进度，保证后续退火曲线完整平滑
        progress = (current_epoch - self.warmup_epochs) / max(1, (self.total_epochs - self.warmup_epochs))
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        # 🔥 修改 3：移除 if current_epoch < 3: return，让热身期也能执行 AGC 和低噪声注入

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC (自适应梯度裁剪)
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印 (🔥 优化了 Log，可以直接看出目前是热身还是退火)
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            phase = "🔥 Warmup" if current_epoch < self.warmup_epochs else "📉 Annealing"
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} ({phase}) | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=10.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp final: 修正")
print("✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 + 前3轮低噪声热身")

if os.path.exists(PRETRAINED_GC10):
    trainer_rescue = Trainer_Rescue(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 50,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': 'Final',
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
    })
    trainer_rescue.train()
else:
    print("❌ 没找到 Exp 13 的权重，无法执行拯救计划。")

🚀 开始 Exp final: 修正
✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 + 前3轮低噪声热身
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False

In [1]:
import os
import time
import psutil
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import numpy as np
import math

# ================= 1. 噪声退火引擎 (支持 ρ 参数化) =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=10.0, strategy='uniform', epochs=30, warmup_epochs=3, warmup_rho=0.15):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy 
        self.total_epochs = epochs
        
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))
        
        # 热身参数
        self.warmup_epochs = warmup_epochs 
        self.warmup_rho = warmup_rho 

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 前三轮热身期，使用稳定的低噪声系数 ρ
        if current_epoch < self.warmup_epochs:
            return self.warmup_rho 

        # 正常退火期：从 1.0 降到 0.1
        min_decay = 0.1 
        progress = (current_epoch - self.warmup_epochs) / max(1, (self.total_epochs - self.warmup_epochs))
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC (自适应梯度裁剪)
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 策略计算
        current_device = next(self.model.parameters()).device
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 初始化引擎
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(
            model, epsilon=10.0, strategy='uniform', epochs=self.epochs, 
            warmup_epochs=3, warmup_rho=0.15
        )
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)


# ================= 3. 性能监控 Callbacks =================
def on_train_epoch_start(trainer):
    """每轮开始时记录时间"""
    trainer.custom_epoch_start_time = time.time()

def on_train_epoch_end(trainer):
    """每轮结束时计算耗时与资源"""
    epoch_time = time.time() - getattr(trainer, 'custom_epoch_start_time', time.time())
    
    # 获取 CPU 和 内存信息
    cpu_usage = psutil.cpu_percent(interval=None) # 当前 CPU 占用率
    mem_info = psutil.virtual_memory()
    mem_used_gb = mem_info.used / (1024 ** 3)
    mem_total_gb = mem_info.total / (1024 ** 3)
    mem_percent = mem_info.percent
    
    phase = "🔥 Warmup (低噪声)" if trainer.epoch < trainer.privacy_engine.warmup_epochs else "📉 Annealing (退火)"
    
    print("\n" + "="*60)
    print(f"📊 [性能报告] Epoch {trainer.epoch + 1}/{trainer.epochs} | 阶段: {phase}")
    print(f"⏱️  本轮耗时: {epoch_time:.2f} 秒 ({epoch_time/60:.2f} 分钟)")
    print(f"💻 CPU 使用率: {cpu_usage}%")
    print(f"🧠 内存使用率: {mem_percent}% ({mem_used_gb:.2f} GB / {mem_total_gb:.2f} GB)")
    print("="*60 + "\n")


# ================= 4. 执行测试 =================
if __name__ == "__main__":
    FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
    PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

    print("🚀 开始 Exp 18: 龟裂拯救计划 (CPU 性能测试版)...")

    if os.path.exists(PRETRAINED_GC10):
        # 记录全局总时间
        total_start_time = time.time()
        
        trainer_rescue = Trainer_Rescue(overrides={
            'model': PRETRAINED_GC10,
            'data': FULL_YAML,
            'epochs': 50,            # 💡 测试 2 轮即可，足以看清 Warmup 的表现
            'batch': 4,             # 💡 减小 Batch Size
            'imgsz': 640,
            'project': 'result_exp1_cpu_test',
            'name': '18_Rescue_CPU',
            'device': 'cpu',        # 🔥 强制 CPU
            'workers': 0,           # 🔥 单进程，防卡死
            'exist_ok': True,
            'freeze': 0
        })
        
        # 将我们写的性能监控函数挂载到 YOLO 引擎上
        trainer_rescue.add_callback("on_train_epoch_start", on_train_epoch_start)
        trainer_rescue.add_callback("on_train_epoch_end", on_train_epoch_end)

        # 开始训练
        trainer_rescue.train()
        
        total_time = time.time() - total_start_time
        print(f"✅ 测试完成！总共耗时: {total_time:.2f} 秒 ({total_time/60:.2f} 分钟)")
    else:
        print("❌ 没找到 Exp 13 的权重，请检查路径。")


libgomp: Invalid value for environment variable OMP_NUM_THREADS


🚀 开始 Exp 18: 龟裂拯救计划 (CPU 性能测试版)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CPU (Intel Xeon Gold 6430)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=18_Rescue_CPU, nbs=64, nms=Fals

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      44/50         0G      1.709      2.302      1.926          8        640: 100% ━━━━━━━━━━━━ 360/360 2.4it/s 2:30<0.4s

📊 [性能报告] Epoch 44/50 | 阶段: 📉 Annealing (退火)
⏱️  本轮耗时: 149.89 秒 (2.50 分钟)
💻 CPU 使用率: 11.0%
🧠 内存使用率: 5.9% (53.38 GB / 1007.52 GB)

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.4s0.4s
                   all        180        391      0.499      0.545      0.558      0.272

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      45/50         0G      1.695      2.262      1.922          9        640: 100% ━━━━━━━━━━━━ 360/360 2.3it/s 2:35<0.4s

📊 [性能报告] Epoch 45/50 | 阶段: 📉 Annealing (退火)
⏱️  本轮耗时: 155.04 秒 (2.58 分钟)
💻 CPU 使用率: 11.1%
🧠 内存使用率: 6.6% (60.43 GB / 1007.52 GB)

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.3it/s 6.9s0.3s
                   all        180        391      0.508      0.5

In [8]:
import os
import cv2
from ultralytics import YOLO

# ================= 1. 配置路径与参数 =================
# 替换为你刚才训练好的/表现最好的权重路径
WEIGHT_PATH = "/root/autodl-tmp/exp1/result_exp1/Final_v1/weights/best.pt" 
# 你的测试集图片文件夹路径 (用验证集或测试集都可以)
TEST_IMAGES_DIR = "/root/autodl-tmp/exp1/NEU_DET_YOLO/images/val"  
# 结果保存文件夹 (专门存放用于论文的图片)
OUTPUT_DIR = "./paper_visualizations"

CONF_THRESHOLD = 0.60  # 🔥 核心要求：置信度大于 70%

# 论文画图专属样式调整 (让框和字体更清晰美观)
LINE_WIDTH = 2   # 边框粗细
FONT_SIZE = 4    # 标签字体大小 (根据图片分辨率可适当调整)

# ================= 2. 准备工作 =================
os.makedirs(OUTPUT_DIR, exist_ok=True)
model = YOLO(WEIGHT_PATH)

class_names = model.names  # 获取模型所有的类别字典，例如 {0: 'crazing', 1: 'inclusion', ...}
target_total_classes = len(class_names)
found_classes = set()      # 用于记录已经保存过展示图的类别

print(f"🚀 开始进行论文制图推理...")
print(f"🎯 目标：为 {target_total_classes} 个类别各寻找一张 conf > {CONF_THRESHOLD} 的展示图。")

# ================= 3. 开始扫描与保存 =================
for img_name in os.listdir(TEST_IMAGES_DIR):
    # 找齐所有类别就可以提前结束了
    if len(found_classes) == target_total_classes:
        break
        
    if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue
        
    img_path = os.path.join(TEST_IMAGES_DIR, img_name)
    
    # 执行推理，直接在这里卡死置信度阈值 conf=0.70
    results = model.predict(img_path, conf=CONF_THRESHOLD, verbose=False)
    result = results[0] # 取单张图片的结果
    
    # 获取这张图中被检测到的所有类别 ID
    # result.boxes.cls 是 tensor，转为 numpy 方便处理
    if len(result.boxes) == 0:
        continue # 这张图里没有置信度大于 0.7 的目标，跳过
        
    detected_classes = result.boxes.cls.cpu().numpy().astype(int)
    
    # 检查是否有我们还没保存过的类别
    for cls_id in detected_classes:
        cls_name = class_names[cls_id]
        
        if cls_name not in found_classes:
            print(f"✅ 找到类别 '{cls_name}' 的高质量样本: {img_name}")
            
            # 🔥 论文级绘图：使用 result.plot() 生成带框的图片矩阵
            # 参数调整：开启 labels 和 conf，设置线宽和字体
            annotated_frame = result.plot(
                line_width=LINE_WIDTH, 
                font_size=FONT_SIZE, 
                conf=False,     # 在框上显示置信度
                labels=True    # 在框上显示类别名
            )
            
            # 存为文件，命名直接用类别名，方便你写论文时插入
            save_path = os.path.join(OUTPUT_DIR, f"demo_{cls_name}_1.jpg")
            cv2.imwrite(save_path, annotated_frame)
            
            # 记录已找到
            found_classes.add(cls_name)
            
            # 只要这张图贡献了一个新类别的展示，就 break 跳出内部循环，
            # 看下一张图，保证论文里的图呈现出多样性 (而不是一张图上框了四五个类别)
            break 

# ================= 4. 结果总结 =================
print("\n" + "="*40)
print(f"🎉 采集完成！图片已保存在: {OUTPUT_DIR}")
print(f"📊 共集齐 {len(found_classes)}/{target_total_classes} 个类别的展示图。")

if len(found_classes) < target_total_classes:
    missing = set(class_names.values()) - found_classes
    print(f"⚠️ 注意，以下类别没有找到置信度 > {CONF_THRESHOLD} 的图片: {missing}")
    print("💡 建议：可以稍微降低 CONF_THRESHOLD，或者检查模型对该类别的检测效果。")
print("="*40)

🚀 开始进行论文制图推理...
🎯 目标：为 6 个类别各寻找一张 conf > 0.6 的展示图。
✅ 找到类别 'patches' 的高质量样本: patches_137.jpg
✅ 找到类别 'scratches' 的高质量样本: scratches_265.jpg
✅ 找到类别 'pitted_surface' 的高质量样本: pitted_surface_288.jpg
✅ 找到类别 'inclusion' 的高质量样本: inclusion_23.jpg
✅ 找到类别 'rolled-in_scale' 的高质量样本: rolled-in_scale_51.jpg

🎉 采集完成！图片已保存在: ./paper_visualizations
📊 共集齐 5/6 个类别的展示图。
⚠️ 注意，以下类别没有找到置信度 > 0.6 的图片: {'crazing'}
💡 建议：可以稍微降低 CONF_THRESHOLD，或者检查模型对该类别的检测效果。
